In [1]:
from pynq import Overlay
from pynq import MMIO
import numpy as np
import time
import struct

import cv2

In [2]:
def quantize_8bit(x):
    # PyTorch tensor를 numpy로 변환
    temp = x
    temp = np.floor(temp)  # Round the values ######################################################
    # 9 LSB 제거 (오른쪽 시프트 9)
    output_shifted = np.right_shift(temp.astype(np.int32), 9)
    # 8-bit signed int 범위로 클리핑 (-128 ~ 127)
    quantized = np.clip(output_shifted, -128, 127).astype(np.int8)
    return quantized

In [3]:
import numpy as np

def reshape_input_and_weights(x, weights, fc_key='fc', kernel_size=3, oc_size=10):
    # Flatten input
    input_flat = x.flatten()
    
    # Calculate padding length
    total_length = np.ceil(len(input_flat) / (kernel_size * kernel_size * 8)) * (kernel_size * kernel_size * 8)
    pad_length = int(total_length - len(input_flat))
    
    # Pad input
    input_padded = np.pad(input_flat, (0, pad_length), mode='constant', constant_values=0)
    
    # Reshape input
    ic = int(total_length // (kernel_size * kernel_size))
    input_reshaped = input_padded.reshape(ic, kernel_size, kernel_size)
    
    # Load weight data
    weight_data = weights[fc_key]
    
    # Initialize reshaped weights array
    weight_reshaped = np.zeros((oc_size, ic, kernel_size, kernel_size), dtype=weight_data.dtype)
    
    # Reshape weights for each output channel
    for oc in range(oc_size):
        weight_flat = weight_data[oc]
        weight_padded = np.pad(weight_flat, (0, pad_length), mode='constant', constant_values=0)
        weight_oc_reshaped = weight_padded.reshape(ic, kernel_size, kernel_size)
        weight_reshaped[oc] = weight_oc_reshaped
    
    return input_reshaped, weight_reshaped

In [4]:
def conv2d_IN_HW(i_act_tile, weight_tile, tile_oc_act_flat, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range):
    stride=1
    # input_ch, tile_h_input, tile_w_input = i_act_tile.shape
    # oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
    # oc_range_out, h_range, w_range = tile_o_act.shape

    for toc in range(oc_range):
        for toh in range(h_range):
            h_start = toh * stride
            for tow in range(w_range):
                w_start = tow * stride
                temp = 0
                for ic in range(input_ch):
                    for kh in range(kernel_h):
                        for kw in range(kernel_w):
                            val = i_act_tile[ic, h_start + kh, w_start + kw].astype(np.float32)
                            wgt = weight_tile[toc, ic, kh, kw].astype(np.float32)
                            temp += val * wgt
                temp = quantize_8bit(temp)
                tile_oc_act_flat[toc*h_range*w_range + toh*w_range + tow] = temp
    return tile_oc_act_flat

In [5]:
def conv2d_HW(i_act, weight, stride=1, tile_h=8, tile_w=8, tile_oc=8, padding=0):
    """
    conv2d (9 for loops) with tiling.
    - i_act: shape (input_ch, input_h, input_w)
    - weight: shape (output_ch, input_ch, kernel_h, kernel_w)
    """
    input_ch, input_h, input_w = i_act.shape
    output_ch, _, kernel_h, kernel_w = weight.shape

    # 출력 feature map 크기 계산
    output_h = (input_h - kernel_h) // stride + 1
    output_w = (input_w - kernel_w) // stride + 1
    o_act = np.zeros((output_ch, output_h, output_w)).astype(np.float32)

    # 타일 단위 반복
    for oh in range(0, output_h, tile_h):
        for ow in range(0, output_w, tile_w):
            for oc in range(0, output_ch, tile_oc):
                h_range = min(tile_h, output_h - oh)
                w_range = min(tile_w, output_w - ow)
                oc_range = min(tile_oc, output_ch - oc)

                # 타일 단위로 i_act와 weight 슬라이싱
                h_start = oh * stride
                w_start = ow * stride
                    
                h_end = h_start + h_range * stride + kernel_h - 1
                w_end = w_start + w_range * stride + kernel_w - 1
                
                i_act_tile = i_act[:, h_start:h_end, w_start:w_end]
                weight_tile = weight[oc:oc + oc_range, :, :, :]

                tile_oc_act_flat = np.zeros(oc_range * h_range * w_range).astype(np.float32)
                tile_o_act = np.zeros((oc_range, h_range, w_range)).astype(np.float32)
                
                input_ch, tile_h_input, tile_w_input = i_act_tile.shape
                oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
                oc_range_out, h_range, w_range = tile_o_act.shape

                # convolution 연산
                tile_oc_act_flat = conv2d_IN_HW(i_act_tile, weight_tile, tile_oc_act_flat, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range)
                tile_o_act = tile_oc_act_flat.reshape(oc_range, h_range, w_range)
                # print("tile_o_act shape:", tile_o_act.shape)  # Debugging output

                o_act[oc:oc + oc_range, oh:oh + h_range, ow:ow + w_range] = tile_o_act

    return o_act

In [6]:
def fc_HW(i_act, weight, stride=1, tile_h=3, tile_w=3, tile_oc=1, padding=0):
#     # 입력 데이터를 flatten: (12800,)
#     input_flat = i_act.flatten()
#     print("fc_input_flat_length: ", len(input_flat))

#     # 필요한 패딩 길이 계산: 12800을 72로 나누어 떨어지게 하기 위해 ceil(12800/72)*72 = 12816
#     total_length = np.ceil(len(input_flat) / 72) * 72
#     pad_length = int(total_length - len(input_flat))  # 16

#     # 입력 데이터 패딩: (12807 -> 12816,)
#     input_padded = np.pad(input_flat, (0, pad_length), mode='constant', constant_values=0)

#     # reshape: (1423 -> 1424, 3, 3)         # IC=1423 -> 1424, H=3, W=3
#     ic = int(total_length // 9)     # 1423 -> 1424
#     input_reshaped = input_padded.reshape(ic, 3, 3)

#     print("INPUT shaping 이후 data 형태:", input_reshaped.shape)

#     # 가중치 데이터 로드: (10, 12800)
#     weight_data = weight

#     # 결과 가중치 배열 초기화: (10, 1423 -> 1424, 3, 3)  # OC=10, IC=1423 -> 1424, H=3, W=3
#     weight_reshaped = np.zeros((10, ic, 3, 3), dtype=weight_data.dtype)

#     print("WEIGHT shaping 이후 data 형태:", weight_reshaped.shape)

#     # 각 OC에 대해 패딩 및 reshape
#     for oc in range(10):
#         weight_flat = weight_data[oc]
#         weight_padded = np.pad(weight_flat, (0, pad_length), mode='constant', constant_values=0)
#         weight_oc_reshaped = weight_padded.reshape(ic, 3, 3)
#         weight_reshaped[oc] = weight_oc_reshaped
    
    """
    Implement FC with conv2d (9 for loops) with tiling.
    - i_act: shape (input_ch, input_h, input_w)
    - weight: shape (output_ch, input_ch, kernel_h, kernel_w)
    """
    input_ch, input_h, input_w = i_act.shape
    output_ch, _, kernel_h, kernel_w = weight.shape
    
    # 출력 feature map 크기 계산
    output_h = (input_h - kernel_h) // stride + 1
    output_w = (input_w - kernel_w) // stride + 1
    o_act = np.zeros((output_ch, output_h, output_w))

    # 타일 단위 반복
    for oh in range(0, output_h, tile_h):
        for ow in range(0, output_w, tile_w):
            for oc in range(0, output_ch, tile_oc):
                h_range = min(tile_h, output_h - oh)
                w_range = min(tile_w, output_w - ow)
                oc_range = min(tile_oc, output_ch - oc)

                # 타일 단위로 i_act와 weight 슬라이싱
                h_start = oh * stride
                w_start = ow * stride
                    
                h_end = h_start + h_range * stride + kernel_h - 1
                w_end = w_start + w_range * stride + kernel_w - 1
                
                i_act_tile = i_act[:, h_start:h_end, w_start:w_end]
                weight_tile = weight[oc:oc + oc_range, :, :, :]

                tile_o_act = np.zeros((oc_range, h_range, w_range))
                
                input_ch, tile_h_input, tile_w_input = i_act_tile.shape
                oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
                oc_range_out, h_range, w_range = tile_o_act.shape
                
                # convolution 연산
                tile_o_act = conv2d_IN_HW(i_act_tile, weight_tile, tile_o_act, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range)

                o_act[oc:oc + oc_range, oh:oh + h_range, ow:ow + w_range] = tile_o_act

    return o_act

In [7]:
def leaky_relu_loop(input_arr, alpha=0.25):
    """
    Leaky ReLU.
    - input_arr: shape (ch, h, w)
    """
    output = np.zeros_like(input_arr).astype(np.float32)
    ch, h, w = input_arr.shape
    for c in range(ch):
        for i in range(h):
            for j in range(w):
                x = input_arr[c, i, j].astype(np.float32)
                output[c, i, j] = x if x > 0 else alpha * x
    return output

In [8]:
def maxpool_loop(input_arr, pool_size=2, stride=2):
    """
    Max pooling
    - input_arr: shape (ch, h, w)
    """
    ch, h, w = input_arr.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((ch, out_h, out_w)).astype(np.float32)
    for c in range(ch):
        for oh in range(out_h):
            h_start = oh * stride
            for ow in range(out_w):
                w_start = ow * stride
                max_val = -np.inf
                for ph in range(pool_size):
                    for pw in range(pool_size):
                        val = input_arr[c, h_start + ph, w_start + pw].astype(np.float32)
                        if val > max_val:
                            max_val = val
                output[c, oh, ow] = max_val
    return output

In [15]:
class NPUDriver:
    #####################################################
    def __init__(self, bitfile_path):
        self.hw = Overlay(bitfile_path)
        self.csr = self.hw.csr_0.mmio.array        #         self.csr = self.hw.csr_0
        self.imem = self.hw.INPUT_MEM.mmio.array
        self.omem = self.hw.OUT_MEM.mmio.array
        self.wmem = self.hw.WEIGHT_MEM.mmio.array
        print(f"IMEM Shape: {self.imem.shape}")
        print(f"WMEM Shape: {self.wmem.shape}")
        print(f"OMEM Shape: {self.omem.shape}")

#         self.csr = self.hw.csr_0.mmio.array
#         self.imem = self.hw.INPUT_MEM.mmio.array
#         self.omem = self.hw.OUT_MEM.mmio.array
#         self.wmem = self.hw.WEIGHT_MEM.mmio.array

#         print("Type of self.csr:", type(self.csr))
#         print("Attributes of self.csr:", dir(self.csr))
#         print(help(self.csr))
    #####################################################

    def write_csr(self, address, value):
        """CSR write"""
        ######################################################################
#         print(f"CSR Write: Addr {hex(address)}, Value {value}")
        ######################################################################
        self.csr[address//4] = value                              # self.csr.write(address, value) 

    def read_csr(self, address):
        return self.csr[address//4]                            # self.csr.write(address, value) 

    def config_layer(self, kw , kh, ic, input_w, input_h, oc):
        """레이어 config 설정"""
        # 공통 config
        self.write_csr(0x08, kw)
        self.write_csr(0x0C, kh)
        self.write_csr(0x10, ic)
        self.write_csr(0x14, input_w)
        self.write_csr(0x18, input_h)
        self.write_csr(0x1C, oc)
        ######################################################################
#         print("################################################################################")
#         print("Later Info-> KW: ", self.read_csr(0x08), "    KH: ", self.read_csr(0x0C), "  IC: ", self.read_csr(0x10))
#         print("             IMG_W: ", self.read_csr(0x14), "IMG_H: ", self.read_csr(0x18), "OC: ", self.read_csr(0x1C))
#         print("################################################################################")
        ######################################################################

    def load_data(self, input_data, weight_data):
########################################################################################################################
        """데이터 로드: WMEM (weight), IMEM (input) -> flatten 시켜서"""
#         self.imem = input_data.astype(np.float32).ravel()
#         self.wmem = weight_data.astype(np.float32).ravel()
#         self.imem[0:input_data.size] = input_data.astype(np.float32).ravel()
#         self.wmem[0:weight_data.size] = weight_data.astype(np.float32).ravel()
        TOTAL_SIZE = 16384
    
        input_data_size = input_data.size
        weight_data_size = weight_data.size
        
        in_data_float32 = input_data.astype(np.float32).ravel()
        weight_data_float32 = weight_data.astype(np.float32).ravel()
        
        # float32를 16진수 정수로 변환
        in_hex_int = [struct.unpack('<I', np.float32(x).tobytes())[0] for x in in_data_float32]
        weight_hex_int = [struct.unpack('<I', np.float32(x).tobytes())[0] for x in weight_data_float32]
        
#         in_hex_values = [np.float32(x).tobytes()[::-1].hex() for x in in_data_float32]
#         weight_hex_values = [np.float32(x).tobytes()[::-1].hex() for x in weight_data_float32]
        
#         self.imem[0:input_data_size] = in_hex_int
#         self.wmem[0:weight_data_size] = weight_hex_int


        # input 데이터 패딩
        padded_in_hex_int = in_hex_int[:input_data_size] + [0] * (TOTAL_SIZE - input_data_size)
        self.imem[0:TOTAL_SIZE] = padded_in_hex_int

        # weight 데이터 패딩
        padded_weight_hex_int = weight_hex_int[:weight_data_size] + [0] * (TOTAL_SIZE - weight_data_size)
        self.wmem[0:TOTAL_SIZE] = padded_weight_hex_int
        
        ######################################################################
#         print(f"IMEM Load: {self.imem.shape}")
#         print(f"INPUT Shape: {input_data.shape}")
# #         print(f"IMEM First 11 elems: {self.imem[0:11]}")
#         print(f"INPUT First 11 elems: {input_data.astype(np.float32).ravel()[0:11]}")
#         print(f"WMEM Load: {self.wmem.shape}")
#         print(f"WEIGHT Shape: {weight_data.shape}")
# #         print(f"WMEM First 11 elems: {self.wmem[0:11]}")
#         print(f"WEIGHT First 11 elems: {weight_data.astype(np.float32).ravel()[0:11]}")
########################################################################################################################

    def get_data(self, num_elements):
        """데이터 가져오기: OMEM (output)"""
        return self.omem[0:num_elements].astype(np.int8)

    def start_npu(self, input_data, weight_data, kw , kh, ic, input_w, input_h, oc):
        """데이터 로드: WMEM (weight), IMEM (input)"""
        self.load_data(input_data, weight_data)
        """레이어 config 설정"""
        self.config_layer(kw , kh, ic, input_w, input_h, oc)
        """Start Signal pulse"""
        self.write_csr(0x04, 1)
        
        ######################################################################
#         print(" ")
#         print("NPU Started")
#         print("Does Start Signal Goes Down? -> Start: ", self.read_csr(0x04))    # self.csr[1]
#         print("NPU Started")
        ######################################################################
        
        while True:
            if (self.read_csr(0x00) == 1): # self.csr[0] == 1
                ######################################################################
#                 print(" ")
#                 print("NPU Done")
                ######################################################################
                break
########################################################################################################################
#         print("################################################################################")
#         print("Later Info-> KW: ", self.csr.read(0x08), "    KH: ", self.csr.read(0x0C), "  IC: ", self.csr.read(0x10))
#         print("             IMG_W: ", self.csr.read(0x14), "IMG_H: ", self.csr.read(0x18), "OC: ", self.csr.read(0x1C))
#         print("################################################################################")
#         return self.get_data(16)
########################################################################################################################
#         print("NPU Output addr 0:1-> ", self.omem[0:1])
        return self.get_data((input_w-2)*(input_h-2)*oc)
    
    def run_conv_2d(self, input_data, weight_data, tile_h=8, tile_w=8, tile_oc=8):
        """
        conv2d (9 for loops) with tiling.
        - i_act: shape (input_ch, input_h, input_w)
        - weight: shape (output_ch, input_ch, kernel_h, kernel_w)
        """
        stride = 1
        input_ch, input_h, input_w = input_data.shape
        output_ch, _, kernel_h, kernel_w = weight_data.shape

        # 출력 feature map 크기 계산
        output_h = (input_h - kernel_h) // stride + 1
        output_w = (input_w - kernel_w) // stride + 1
        o_act = np.zeros((output_ch, output_h, output_w)).astype(np.float32)

        # 타일 단위 반복
        for oh in range(0, output_h, tile_h):
            for ow in range(0, output_w, tile_w):
                for oc in range(0, output_ch, tile_oc):
                    h_range = min(tile_h, output_h - oh)
                    w_range = min(tile_w, output_w - ow)
                    oc_range = min(tile_oc, output_ch - oc)

                    # 타일 단위로 i_act와 weight 슬라이싱
                    h_start = oh * stride
                    w_start = ow * stride
                        
                    h_end = h_start + h_range * stride + kernel_h - 1
                    w_end = w_start + w_range * stride + kernel_w - 1
                    
                    i_act_tile = input_data[:, h_start:h_end, w_start:w_end]
                    weight_tile = weight_data[oc:oc + oc_range, :, :, :]

                    tile_oc_act_flat = np.zeros(oc_range * h_range * w_range).astype(np.float32)
                    tile_o_act = np.zeros((oc_range, h_range, w_range)).astype(np.float32)
                    
                    input_ch, tile_h_input, tile_w_input = i_act_tile.shape
                    oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
                    oc_range_out, h_range, w_range = tile_o_act.shape
                    
                    ########################################################################################################################
                    # convolution 연산
                    tile_oc_act_flat = self.start_npu(i_act_tile, weight_tile, kernel_w, kernel_h, input_ch, tile_w_input, tile_h_input, oc_range)
                    ########################################################################################################################
                    # convolution 연산
#                     tile_oc_act_flat = conv2d_IN_HW(i_act_tile, weight_tile, tile_oc_act_flat, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range)
                    ########################################################################################################################
                    tile_o_act = tile_oc_act_flat.reshape(oc_range, h_range, w_range)
                    
                    o_act[oc:oc + oc_range, oh:oh + h_range, ow:ow + w_range] = tile_o_act

        return o_act

    def run_fc_2d(self, i_act, weight, stride=1, tile_h=3, tile_w=3, tile_oc=1, padding=0):
        """
        Implement FC with conv2d (9 for loops) with tiling.
        - i_act: shape (input_ch, input_h, input_w)
        - weight: shape (output_ch, input_ch, kernel_h, kernel_w)
        """
        input_ch, input_h, input_w = i_act.shape
        output_ch, _, kernel_h, kernel_w = weight.shape

        # 출력 feature map 크기 계산
        output_h = (input_h - kernel_h) // stride + 1
        output_w = (input_w - kernel_w) // stride + 1
        o_act = np.zeros((output_ch, output_h, output_w))
        
        out_addr = 0
        
        # 타일 단위 반복
        for oh in range(0, output_h, tile_h):
            for ow in range(0, output_w, tile_w):
                for oc in range(0, output_ch, tile_oc):
                    h_range = min(tile_h, output_h - oh)
                    w_range = min(tile_w, output_w - ow)
                    oc_range = min(tile_oc, output_ch - oc)

                    # 타일 단위로 i_act와 weight 슬라이싱
                    h_start = oh * stride
                    w_start = ow * stride
                        
                    h_end = h_start + h_range * stride + kernel_h - 1
                    w_end = w_start + w_range * stride + kernel_w - 1

                    i_act_tile = i_act[:, h_start:h_end, w_start:w_end]
                    weight_tile = weight[oc:oc + oc_range, :, :, :]

                    tile_oc_act_flat = np.zeros(oc_range * h_range * w_range).astype(np.float32)
                    tile_o_act = np.zeros((oc_range, h_range, w_range)).astype(np.float32)
                    
                    input_ch, tile_h_input, tile_w_input = i_act_tile.shape
                    oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
                    oc_range_out, h_range, w_range = tile_o_act.shape
                    
                    ########################################################################################################################
                    # convolution 연산
                    tile_oc_act_flat = self.start_npu(i_act_tile, weight_tile, kernel_w, kernel_h, input_ch, tile_w_input, tile_h_input, oc_range)
                    ########################################################################################################################
                    # convolution 연산
#                     tile_oc_act_flat = conv2d_IN_HW(i_act_tile, weight_tile, tile_oc_act_flat, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range)
                    ########################################################################################################################
                    tile_o_act = tile_oc_act_flat.reshape(oc_range, h_range, w_range)

#                     o_act[oc:oc + oc_range, oh:oh + h_range, ow:ow + w_range] = tile_o_act
                    o_act[out_addr] = tile_o_act

                    out_addr += 1

        return o_act


In [29]:
driver = NPUDriver("attempt_20.bit")
# driver = NPUDriver("check_done.bit")

IMEM Shape: (131072,)
WMEM Shape: (262144,)
OMEM Shape: (131072,)


In [30]:
input_image = np.load('npy_files/input.npy')
weights = {
    'conv1': np.load('npy_files/layer1_0_weight.npy'),
    'conv2': np.load('npy_files/layer2_0_weight.npy'),
    'conv3': np.load('npy_files/layer3_0_weight.npy'),
    'conv4': np.load('npy_files/layer4_0_weight.npy'),
    'fc': np.load('npy_files/fc1_weight.npy')
}

In [31]:
def npu_golden_model_HW(input_image, weights):
    """
    Golden Model을 사용한 전체 CNN Forward Propagation.
    - input_image: 형태 (1, 28, 28)의 np.array
    - weights: dict, key 'conv1' (16,1,3,3), 'conv2' (64,16,3,3), 'conv3' (128,64,3,3), 'conv4' (128,128,3,3), 'fc' (12800,10)
    """
    # Conv1 + Leaky ReLU
    x = driver.run_conv_2d(input_image, weights['conv1'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv1 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # Conv2 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv2'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv2 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # Conv3 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv3'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv3 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # Conv4 + Leaky ReLU
    x = driver.run_conv_2d(x, weights['conv4'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv4 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # 최대 풀링
    x = maxpool_loop(x, pool_size=2, stride=2)
    print("Maxpool output shape:", x.shape)  # Debugging output

    input_reshaped, weight_reshaped = reshape_input_and_weights(x, weights)

    # FC
    output = driver.run_fc_2d(input_reshaped, weight_reshaped, tile_h=3, tile_w=3, tile_oc=1)
# 
    return output.reshape(-1)

In [32]:
def npu_golden_model(input_image, weights):
    """
    Golden Model을 사용한 전체 CNN Forward Propagation.
    - input_image: 형태 (1, 28, 28)의 np.array
    - weights: dict, key 'conv1' (16,1,3,3), 'conv2' (64,16,3,3), 'conv3' (128,64,3,3), 'conv4' (128,128,3,3), 'fc' (12800,10)
    """
    # Conv1 + Leaky ReLU
    x = conv2d_HW(input_image, weights['conv1'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv1 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # Conv2 + Leaky ReLU
    x = conv2d_HW(x, weights['conv2'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv2 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # Conv3 + Leaky ReLU
    x = conv2d_HW(x, weights['conv3'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv3 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # Conv4 + Leaky ReLU
    x = conv2d_HW(x, weights['conv4'], tile_h=8, tile_w=8, tile_oc=8)
    print("Conv4 output shape:", x.shape)  # Debugging output
    x = leaky_relu_loop(x)

    # 최대 풀링
    x = maxpool_loop(x, pool_size=2, stride=2)
    print("Maxpool output shape:", x.shape)  # Debugging output

    input_reshaped, weight_reshaped = reshape_input_and_weights(x, weights)

    # FC
    output = fc_HW(input_reshaped, weight_reshaped, tile_h=3, tile_w=3, tile_oc=1)
# 
    return output.reshape(-1)

In [33]:
start_time = time.time()
#############################################################
print("처음은 NPU가 이미지 0 or 1 or 2번 계산")
# output = npu_golden_model_HW(input_image[0], weights)
# print("출력 형태:", output.shape)       # (10,) 이어야 함
# print("출력 값:", output)
# output = npu_golden_model_HW(input_image[1], weights)
# print("출력 형태:", output.shape)       # (10,) 이어야 함
# print("출력 값:", output)
output = npu_golden_model_HW(input_image[2], weights)
print("출력 형태:", output.shape)       # (10,) 이어야 함
print("출력 값:", output)
#############################################################
end_time = time.time()
runtime = end_time - start_time

# print("출력 형태:", output.shape)       # (10,) 이어야 함
# print("출력 값:", output)
print(f"Runtime: {runtime*1000:.3f}ms")

# start_time = time.time()
# #############################################################
# print("이제부터 PS가 이미지 0번 계산")
# output = npu_golden_model(input_image[0], weights)
# #############################################################
# end_time = time.time()
# runtime = end_time - start_time

# print("출력 형태:", output.shape)       # (10,) 이어야 함
# print("출력 값:", output)
# print(f"Runtime: {runtime*1000:.3f}ms")

처음은 NPU가 이미지 0 or 1 or 2번 계산
Conv1 output shape: (8, 26, 26)
Conv2 output shape: (16, 24, 24)
Conv3 output shape: (64, 22, 22)
Conv4 output shape: (128, 20, 20)
Maxpool output shape: (128, 10, 10)
출력 형태: (10,)
출력 값: [ -8.  75.   6. -40.  25. -10. -24. -33.  22. -31.]
Runtime: 64667.883ms


In [28]:
answer = np.load('npy_files/output.npy')
print("정답 형태:", answer[0].shape)
print("정답 레이블:", answer[0])  # 정답 레이블 확인
print("정답 레이블:", answer[1])  # 정답 레이블 확인
print("정답 레이블:", answer[2])  # 정답 레이블 확인

정답 형태: (10,)
정답 레이블: [ -53.  -45.   -8.   53. -102.  -51. -128.  127.   14.   14.]
정답 레이블: [  39.   28.  127.  -24.  -35.  -77.   28. -128.   11.  -52.]
정답 레이블: [ -8.  75.   6. -40.  25. -10. -24. -33.  22. -31.]


In [22]:
### TEST ###
import numpy as np

driver = NPUDriver("attempt_20.bit")
# driver.load_data(input_data, weight)

####################### Check Data to hex ###########################
# data_float32 = input_data.astype(np.float32).ravel()
# # 16진수로 변환
# hex_values = [np.float32(x).tobytes()[::-1].hex() for x in data_float32]

# # 결과 출력
# for i, (val, hex_val) in enumerate(zip(data_float32, hex_values)):
#     print(f"Value {i}: {val} -> Hex: {hex_val}")
#####################################################################

####################### Check CSR Operation ###########################
# driver = NPUDriver("check_done.bit")
# driver = NPUDriver("check_csr_6.bit")

# output = driver.start_npu(input_data, weight, 3 , 3, 8, 3, 3, 8)
# output = driver.start_npu(input_data, weight, 3, 3, 8, 3, 3, 3)
#####################################################################

####################### Test With 2 different Tiles ###########################
# start_npu(self, input_data, weight_data, kw , kh, ic, input_w, input_h, oc):

# input_data = np.load('i_act_tile_2.npy')
# weight = np.load('weight_tile_2.npy')
# output = driver.start_npu(input_data, weight, 3 , 3, 64, 6, 10, 8)

# input_data = np.load('i_act_tile_3.npy')
# weight = np.load('weight_tile_3.npy')
# output = driver.start_npu(input_data, weight, 3 , 3, 1, 10, 10, 8)
###############################################################################

########################### Test CONV1 Layer ###############################
# input_image = np.load('npy_files/input.npy')
# input_data = input_image[0]
# weight = np.load('npy_files/layer1_0_weight.npy')
# output = driver.run_conv_2d(input_data, weight, tile_h=8, tile_w=8, tile_oc=8)
# np.savetxt('conv1_output.txt', output.flatten(), fmt='%f')
###############################################################################

########################### Test CONV2 Layer ###############################
# input_data = np.load('npy_files/conv1_n_leaky_output.npy')
# weight = np.load('npy_files/layer2_0_weight.npy')
# output = driver.run_conv_2d(input_data, weight, tile_h=8, tile_w=8, tile_oc=8)
# np.savetxt('conv2_output.txt', output.flatten(), fmt='%f')        # .reshape(-1, output.shape[-1])
###############################################################################

########################### Test CONV3 Layer ###############################
# input_data = np.load('npy_files/conv2_n_leaky_output.npy')
# weight = np.load('npy_files/layer3_0_weight.npy')
# output = driver.run_conv_2d(input_data, weight, tile_h=8, tile_w=8, tile_oc=8)
# np.savetxt('conv3_output.txt', output.flatten(), fmt='%f')
###############################################################################

########################### Test CONV4 Layer ###############################
# input_data = np.load('npy_files/conv3_n_leaky_output.npy')
# weight = np.load('npy_files/layer4_0_weight.npy')
# output = driver.run_conv_2d(input_data, weight, tile_h=8, tile_w=8, tile_oc=8)
# np.savetxt('conv4_output.txt', output.flatten(), fmt='%f')                         # .reshape(-1, output.shape[-1])
###############################################################################

########################### Test FC Layer ###############################
x = np.load('npy_files/conv4_n_maxpool_output.npy')
weights_fc = {'fc': np.load('npy_files/fc1_weight.npy')}

input_reshaped, weight_reshaped = reshape_input_and_weights(x, weights_fc)

output = driver.run_fc_2d(input_reshaped, weight_reshaped, tile_h=3, tile_w=3, tile_oc=1).reshape(-1)
###############################################################################


print("출력 형태:", output.shape)                # tile_2의 경우 (8,8,4) 이어야 함
print("출력 값:", output) 
# print("출력 값:", output.astype(np.int8)) 


IMEM Shape: (131072,)
WMEM Shape: (262144,)
OMEM Shape: (131072,)
출력 형태: (10,)
출력 값: [ -53.  -45.   -8.   53. -102.  -51. -128.  127.   14.   14.]


In [23]:
import numpy as np

def run_ps_conv_2d(input_data, weight_data, tile_h=8, tile_w=8, tile_oc=8):
    """
    conv2d (9 for loops) with tiling.
    - i_act: shape (input_ch, input_h, input_w)
    - weight: shape (output_ch, input_ch, kernel_h, kernel_w)
    """
    stride = 1
    input_ch, input_h, input_w = input_data.shape
    output_ch, _, kernel_h, kernel_w = weight_data.shape

    # 출력 feature map 크기 계산
    output_h = (input_h - kernel_h) // stride + 1
    output_w = (input_w - kernel_w) // stride + 1
    o_act = np.zeros((output_ch, output_h, output_w)).astype(np.float32)

    # 타일 단위 반복
    for oh in range(0, output_h, tile_h):
        for ow in range(0, output_w, tile_w):
            for oc in range(0, output_ch, tile_oc):
                h_range = min(tile_h, output_h - oh)
                w_range = min(tile_w, output_w - ow)
                oc_range = min(tile_oc, output_ch - oc)

                # 타일 단위로 i_act와 weight 슬라이싱
                h_start = oh * stride
                w_start = ow * stride

                h_end = h_start + h_range * stride + kernel_h - 1
                w_end = w_start + w_range * stride + kernel_w - 1

                i_act_tile = input_data[:, h_start:h_end, w_start:w_end]
                weight_tile = weight_data[oc:oc + oc_range, :, :, :]

                tile_oc_act_flat = np.zeros(oc_range * h_range * w_range).astype(np.float32)
                tile_o_act = np.zeros((oc_range, h_range, w_range)).astype(np.float32)

                input_ch, tile_h_input, tile_w_input = i_act_tile.shape
                oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
                oc_range_out, h_range, w_range = tile_o_act.shape

                ########################################################################################################################
                # convolution 연산
#                 tile_oc_act_flat = self.start_npu(i_act_tile, weight_tile, kernel_w, kernel_h, input_ch, tile_w_input, tile_h_input, oc_range)
                ########################################################################################################################
                # convolution 연산
                tile_oc_act_flat = conv2d_IN_HW(i_act_tile, weight_tile, tile_oc_act_flat, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range)
                ########################################################################################################################
                tile_o_act = tile_oc_act_flat.reshape(oc_range, h_range, w_range)

                o_act[oc:oc + oc_range, oh:oh + h_range, ow:ow + w_range] = tile_o_act

    return o_act

In [24]:
# test_array = [1, 1, 1, 1]
# test_array_np = np.array(test_array)
# print(test_array_np.shape)
# print(input_data.shape)
# input_data_flatten = input_data.ravel()
# print(input_data_flatten.shape)

####################################################################################
# print(input_data.shape)
# print(weight.shape)
# print(input_data.flatten()[0:11])
# print(weight[0][0])
####################################################################################
# output_ps = run_ps_conv_2d(input_data, weight, tile_h=8, tile_w=8, tile_oc=8)        # .flatten()
# np.savetxt('conv1_output_ps.txt', output_ps.flatten(), fmt='%f')
####################################################################################
output_ps = fc_HW(input_reshaped, weight_reshaped, tile_h=3, tile_w=3, tile_oc=1).reshape(-1)
####################################################################################
print("출력 형태:", output_ps.shape)       # (8,8,4) 이어야 함
print("출력 값:", output_ps) 

출력 형태: (10,)
출력 값: [ -53.  -45.   -8.   53. -102.  -51. -128.  127.   14.   14.]


In [25]:
def compare_arrays(output_ps, output):
    # 정확한 일치 확인
    if np.array_equal(output_ps, output):
        print("두 배열은 정확히 동일합니다.")
    else:
        print("두 배열은 정확히 동일하지 않습니다.")
            
        # 차이점 분석
        diff_mask = np.abs(output_ps - output) > 0
        num_differences = np.sum(diff_mask)
        print(f"총 {num_differences}개의 요소가 다릅니다.")

        if num_differences > 0:
            # Flatten된 배열로 변환
            flat_output_ps = output_ps.flatten()
            flat_output = output.flatten()
            flat_diff_mask = diff_mask.flatten()
            
            # 차이가 있는 1차원 인덱스
            flat_diff_indices = np.where(flat_diff_mask)[0]
            print("다른 요소의 1차원 인덱스:", flat_diff_indices.tolist())
            
            # 각 배열에서 다른 요소 값 출력
            print("output_ps의 값:", flat_output_ps[flat_diff_indices])
            print("output의 값:", flat_output[flat_diff_indices])
#             # 차이가 있는 인덱스
#             diff_indices = np.where(diff_mask)
#             print("다른 요소의 인덱스:", list(zip(diff_indices[0], diff_indices[1])))

#             # 각 배열에서 다른 요소 값 출력
#             print("output_ps의 값:", output_ps[diff_indices])
#             print("output의 값:", output[diff_indices])

In [26]:
compare_arrays(output_ps, output)

두 배열은 정확히 동일합니다.
